In [ ]:
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from iso639 import languages
from sklearn.linear_model import LinearRegression
from scipy import stats

In [ ]:
base_path = "results"
score_path = "results"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 1
models = [
          #"BAAI__bge-m3",
          #"codefuse-ai__F2LLM-v2-4B",
          #"google__embeddinggemma-300m",
          #"intfloat__multilingual-e5-large-instruct",
          #"microsoft__harrier-oss-v1-0.6b",
          #"Octen__Octen-Embedding-8B",
          "Qwen__Qwen3-Embedding-0.6B",
          "__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model",
          #"Qwen__Qwen3-Embedding-4B",
          #"ibm-granite__granite-embedding-311m-multilingual-r2",
          ]
filter_prompts=False
#dataset = "mteb__tatoeba-bitext-mining:vie-eng" 
dataset = "mteb__ARCChallenge" 
#dataset = "squad" 
split = "test"
score= f"recall@{k}" # "ndcg@10"#"F1" #"Accuracy" #"V-score" # "average_precision" 
subsplit=""
path = lambda model: f"{base_path}/{model}/{dataset.replace(':','_')}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset.replace(':','_')}/{split}/{template}_template/"

In [ ]:
# Retrieval prompt
prompts_retrieval = ["Given a question, retrieve the passage that best answers it.",
            "Retrieve.",
            "Find the most relevant passage that directly answers the question.",
            "Given a question, find a related document.",
            "Retrieve the answer to the question.",
            "Retrieve text based on user query.",
            "Given a question, retrieve Wikipedia passages that answer the question.",
            ]
prompts_sts = ["Retrieve semantically similar text.",
            "Retrieve a similar passage.",
            "Represent this sentence for a natural language understanding task.",
            "Group passages based on semantic similarity.",
            ]
prompts_lang = lambda lang: ["Retrieve parallel sentences.",
            f"Retrieve the corresponding translation in {lang}.",
            f"Given an English sentence, find its translation in {lang}.",
            f"Retrieve parellel sentences in {lang}.",
            f"Translate to {lang}.",
            "Find a sentence that has similar meaning.",
            "Retrieve the corresponding translation.",
            "Given an English sentence, find its translation.",
            "Retrieve parellel sentences.",
            "Translate.",
            "Find a sentence that has similar meaning.",
            ]
prompts_keysmash = ["asdfjkl qpwoeiru zxcvbnm",
            "hgJKSbf oiawnef LKJHDS kdjfbs",
            "!!!! ??? ### @@@",
            "EMPTY"]
prompts_finetune = [ "Given a question, retrieve Wikipedia passages that answer the question.",
            "Given a question, retrieve questions that are semantically equivalent to the given question."]

if ":" in dataset:
    l = dataset.split(":")[-1].split("-")[0]
    try:
        lang = languages.get(part2t=l).name
    except KeyError as err:
        if l == "cmn":
            lang = "Mandarin Chinese"
        else:
            raise KeyError(f"Cannot resolve {l} with iso639 in prompts") from err
    prompts_appropriate = prompts_lang(lang)
else:
    prompts_appropriate = prompts_retrieval + ["Given a question, retrieve Wikipedia passages that answer the question."]
    # added also the second finetuning prompt here.

In [ ]:

def construct_df(model, show=False):
    scores_path= path_scores(model)+f"eval@1_2_5_10.json"
    with open(scores_path) as f:
        scores = json.load(f)
    scores_path2= path_scores(model)+f"eval@1_2_5_10_with_distractors.json"
    with open(scores_path2) as f:
        scores2 = json.load(f)
    with open(path(model)+f"prompt_geometry_10nn_1_distractor_and_1_false_positive.json") as f:
        data1 = json.load(f)
    df_scores = pd.DataFrame.from_dict(scores).T
    df_scores2 = pd.DataFrame.from_dict(scores2).T
    # there is one duplicate prompt in Tatoeba specifically, drop it here
    df_scores = df_scores.drop_duplicates(subset="prompt_text")
    df_scores2 = df_scores2.drop_duplicates(subset="prompt_text")
    df_all_scores = df_scores.merge(df_scores2, suffixes=("", "_distracted"), on='prompt_text')
    df_angle = pd.DataFrame.from_dict(data1).T
    # again, there is the one duplicate
    df_angle = df_angle.drop_duplicates(subset="prompt_text")
    df = df_all_scores.merge(df_angle, on='prompt_text')
    if filter_prompts:
        prompts = prompts_appropriate+prompts_finetune+prompts_sts+["NO_PROMPT", "EMPTY"]
        df = df[df["prompt_text"].isin(prompts)]
    if show: display(df.head())
    return df

#_ = construct_df(models[-1], show=True)

In [ ]:

def plot(df, x, y="score", colors=None, sizes=None, title="", legend_title=None, x_min=None, x_max=None, y_min=None, y_max=None, return_fig=False, add_line=False):
    if colors is None:
        colors = y
    if sizes is None:
        sizes = y
    
    # legend title that explains formatting
    if legend_title is None:
        legend_title = f"colors:{colors}, size:{sizes}"

    # Normalize scores for marker size
    min_size, max_size = 10, 30
    try:
        # parse the value from dictionary
        df["sizes"] = df[sizes].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
        ranks = df["sizes"].rank(method='average')
    except:   # for non-dict format: i.e. prompt_label or score
        ranks = df[sizes].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )
    #print(df["sizes"])

    #print(marker_sizes)
    y_vals = df[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    #y_err = df[y].apply(lambda d: float(d['std']) if isinstance(d, dict) else float(d[1]) if isinstance(d, list) else 0.0)
    
    y_err = []
    for line in df[y]:
        if isinstance(line, dict):
            if "std" in line.keys():
                y_err.append(float(line["std"]))
            elif "confidence_interval" in line.keys():
                y_err.append(float(max(line["confidence_interval"])))
            else:
                y_err.append(0.0)
        else:
            y_err.append(0.0)
        

    x_vals = df[x].apply(lambda d: float(d['mean']))
    x_err  = df[x].apply(lambda d: float(d['std']))
    
    symbols = ["x" if p in prompts_appropriate else "square" if p in prompts_keysmash else "circle" for p in df["prompt_text"]]

    # Create figure
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            mode='markers',
            x=x_vals,
            y=y_vals,
            #error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            #error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
                symbol=symbols,
            #    colorscale='Cividis',
                color=df[colors].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d)),
                colorbar=dict(title=f"C:{colors} S:{sizes}"),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (distance):</b> %{x:.4f}<br>'
                '<b>Y (score):</b> %{y:.4f}<br>'
            ),
        ),
    )


    # Add one trace per alpha value

    fig.update_layout(
        title = title,
        xaxis_title=x,#'Cos-distance compared to Q-A line',
        yaxis_title=y,#'Prompt performance',
        height=600,
        width=1000,
        template="none",
    )
    fig.update_layout(legend_title_text=legend_title)
    fig.update_xaxes(range=[x_min, x_max])
    fig.update_yaxes(range=[y_min, y_max], autorange=False)

    if add_line:
        for (x0, y0), (x1, y1) in add_line:
            fig.add_shape(
                type="line",
                x0=x0, y0=y0,                  # Starting point (adjust to match your axis limits if needed)
                x1=x1, y1=y1,                  # Ending point
                xref="x", yref="y",          # Binds coordinates to the data scale
                xsizemode="scaled",          # Keeps the slope relative to the axes
                ysizemode="scaled",
                line=dict(color="Red", width=2, dash="dash") # Optional styling
            )
    include = np.where(df["prompt_text"] != "NO_PROMPT")[0]
    #print(np.array(x_vals)[include])
    print(stats.pearsonr(np.array(x_vals)[include], np.array(y_vals)[include]))
    if return_fig:
        return fig
    else:
        fig.show()

In [ ]:
dfs = {}
for m in models:
    try:
        dfs[m] = construct_df(m)
    except Exception as e:
        print(f"Cannot construct results for {m}")
        print(e)

In [ ]:
print(prompts_appropriate)

['Given a question, retrieve the passage that best answers it.', 'Retrieve.', 'Find the most relevant passage that directly answers the question.', 'Given a question, find a related document.', 'Retrieve the answer to the question.', 'Retrieve text based on user query.', 'Given a question, retrieve Wikipedia passages that answer the question.', 'Given a question, retrieve Wikipedia passages that answer the question.']


In [ ]:
prompt_labels = np.array([0,0,0,0,1,0,1,1,1,0,0]).reshape(-1, 1)
scores = np.array([1,2,1,2,10,4,11,8,5,4,1]).reshape(-1, 1)
lreg = LinearRegression().fit(prompt_labels, scores)
print(lreg.score(prompt_labels, scores))

0.7635434740697898


In [ ]:
# first look if we can separate the "appropriate labels" for others by score
def to_latex_rows(args):
    final=""
    for line in args:
        if isinstance(line, float):
            final+=str(np.round(line, 3)) + " & "
        else:
            final+=str(line) + " & "
    final = final[:-3] + "\\\\"
    print(final)


score_to_analyse = "sim_q2pq"
print(score_to_analyse)
for m in dfs.keys():
    # select the data by model
    dfe = dfs[m]
    scores = np.array([[d["mean"]] for d in dfe[score_to_analyse]])
    prompt_labels = np.array([[int(p in prompts_appropriate)] for p in dfe["prompt_text"]])
    # linear regression
    lreg = LinearRegression().fit(prompt_labels, scores)
    reg_result = lreg.score(prompt_labels, scores)  # r2
    # mann-whitney
    appr = scores[prompt_labels == [1]]
    not_appr = scores[prompt_labels == [0]]
    stat, p = stats.mannwhitneyu(appr, not_appr, alternative='greater')  # one tailed, hypothesis: appr greater
    # this returns U for appr, which is larger hence use the formula below
    r_rb =  (2 * stat) / (len(appr) * len(not_appr)) -1
    # top k performing prompts
    k = len(appr)
    scores_flat = scores.flatten()
    prompt_labels_flat = prompt_labels.flatten()
    sorted_scores = np.argsort(scores_flat)[::-1][:k]
    best_prompts = np.array([p for p in dfe["prompt_text"]])[sorted_scores]
    best_prompts_appropriateness = prompt_labels_flat[sorted_scores]
    #print(best_prompts)   # for sanity check
    fraction_of_relevant_in_top = sum(best_prompts_appropriateness)/k
    median_difference = np.median(appr)-np.median(not_appr)
    to_latex_rows([m, fraction_of_relevant_in_top, reg_result, r_rb])#f"{r_rb}{'*' if p < 0.05 else ''}"])
    #print(f"{m}:\n\t{lreg.score(prompt_labels, scores)}\n\tAppr: {np.median(appr)}, Not-Appr: {np.median(not_appr)}\n\t{stat}{'*' if p < 0.05 else ''}\n\t")
    #print(f"{m}:\n\t{lreg.score(prompt_labels, scores)}\n\t{lreg2.score(prompt_labels, scores2)}\n\t{stat}{'*' if p < 0.05 else ''}\n\t{stat2}{'*' if p2 < 0.05 else ''}")
    
    

sim_q2pq
Qwen__Qwen3-Embedding-0.6B & 0.143 & 0.053 & 0.388\\
__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model & 0.143 & 0.02 & 0.136\\


In [ ]:
# max == mean here: only one hard neg
for m in dfs.keys():
    df = dfs[m]
    print(m)
    x = "sim_q2pq_euc" #paraphrase_neg_sim_change_mean" #"hard_neg_angulation_max" #"hard_neg_sim_change_max" #"chord_similarity" #"sim_q2pq"#"sim_improvement" #"paraphrase_neg_sim_change_mean"
    y = "ndcg@10" #"recall@1"
    x_min= None 
    x_max = None 
    y_min=None
    y_max=None
    _= plot(df, x=x, y=y, title=f"{dataset}: {m}", x_min=x_min, x_max = x_max, y_min=y_min, y_max=y_max, return_fig=False)
    #y = f"ndcg@10_distracted"
    #_= plot(df, x=x, y=y, title=f"{dataset}: {m}", x_min=x_min, x_max = x_max, y_min=y_min, y_max=y_max, return_fig=True)



Qwen__Qwen3-Embedding-0.6B
PearsonRResult(statistic=-0.8212554116999479, pvalue=8.696453554528409e-13)


__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model
PearsonRResult(statistic=-0.7584782866348225, pvalue=4.245713524497489e-10)


In [ ]:
for m in dfs.keys():
    df = dfs[m]
    #print(df.columns)
    sim_QA = df["sim_q2a_euc"][0]["mean"]  # this is the max value we can have
    x = "sim_q2pq_euc"#"hard_neg_sim_change_mean"
    y = "sim_pq2a_euc"#"paraphrase_neg_sim_change_mean" 
    sizes=f"recall@10"
    colors = f"recall@10"
    x_min=None
    x_max = None
    y_min=None
    y_max = None
    plot(df, x=x, y=y, sizes=sizes, colors=colors, title=f"{dataset}: {m}", x_min=x_min, x_max = x_max, y_min= y_min, y_max = y_max, add_line=[((0,0), (1,1)), ((sim_QA, 0), (0, sim_QA))])

PearsonRResult(statistic=0.8968525619932244, pvalue=6.5708500271769114e-18)


PearsonRResult(statistic=0.5789666096784942, pvalue=1.6254076937930013e-05)


In [ ]:

def plot_paired_differences(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10"
    ):
    """
    Plot paired (x, y) points from two DataFrames, connected by dashed lines.

    Parameters
    ----------
    df1, df2    : DataFrames with columns 'x' and 'y' (same length, rows are paired)
    hover_col   : Optional column name in both DataFrames to show as hover text
    df1_name    : Legend label for df1 points
    df2_name    : Legend label for df2 points
    df1_color   : Marker color for df1
    df2_color   : Marker color for df2
    line_color  : Color of the dashed connector lines
    """
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))

    # Connectroe lines
    # Interleave (x1, x2, None) for each pair so plotly draws separate segments
    line_x, line_y = [], []
    for x1, x2, y1, y2 in zip(x_vals1, x_vals2, y_vals1, y_vals2):
        line_x += [x1, x2, None]
        line_y += [y1, y2, None]

    fig.add_trace(go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line=dict(dash="dash", color=line_color, width=1.5),
        hoverinfo="skip",
        showlegend=False,
    ))

    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")
    
    # First data
    colors1 = [df1_color if p in prompts_appropriate else "lightblue" for p in df1["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df1["prompt_text"]]
    text1, htemplate1 = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals1, y=y_vals1,
        mode="markers",
        name=df1_name,
        marker=dict(color=colors1, size=10, line=dict(width=1, color="white"), symbol=symbols),
        text=text1,
        hovertemplate=htemplate1,
    ))

    # Second data
    colors2 = [df2_color if p in prompts_appropriate else "lightpink" for p in df2["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df2["prompt_text"]]
    text2, htemplate2 = make_hover(df2, df2_name)
    fig.add_trace(go.Scatter(
        x=x_vals2, y=y_vals2,
        mode="markers",
        name=df2_name,
        marker=dict(color=colors2, size=10, line=dict(width=1, color="white"), symbol=symbols),
        text=text2,
        hovertemplate=htemplate2,
    ))

    fig.update_layout(
        xaxis_title=x,
        yaxis_title=y,
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig


In [ ]:
def plot_difference(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10",
    sizes=None,
    ):
    if sizes is None:
        sizes=y
    
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals_diff = np.array(x_vals2)-np.array(x_vals1)
    y_vals_diff = np.array(y_vals2)-np.array(y_vals1)
    
    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")

    # Normalize scores for marker size
    min_size, max_size = 8, 32
    df1["sizes"] = df1[sizes].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    ranks = df1["sizes"].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )
    # First data
    colors = [df2_color if p in prompts_appropriate else "lightpink" for p in df2["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df2["prompt_text"]]
    text, htemplate = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals_diff, y=y_vals_diff,
        mode="markers",
        name="Difference",
        marker=dict(color=colors, size=marker_sizes, line=dict(width=1, color="white"), symbol=symbols),
        text=text,
        hovertemplate=htemplate,
    ))


    fig.update_layout(
        xaxis_title=f"Difference in {x}",
        yaxis_title=f"Difference in {y}",
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig

In [ ]:
x = "paraphrase_neg_sim_change_mean"
y = f"recall@{5}_distracted"

fig = plot_paired_differences(
                            dfs["Qwen__Qwen3-Embedding-0.6B"], 
                            #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                            dfs["__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model"], 
                            hover_col="prompt_text", 
                            x=x,
                            y=y,
                            df1_name="Qwen3-Embedding-0.6B", 
                            df2_name="Finetuning checkpoint 18k")
fig.show()

fig = plot_difference(
                    dfs["Qwen__Qwen3-Embedding-0.6B"], 
                    #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                    dfs["__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model"],
                    hover_col="prompt_text",
                    x=x,
                    y=y)
fig.show()